# TextTraits Colab One-Shot

Open this notebook in PyCharm 2025.3.2+ and connect it to a Google Colab server. It clones the repo into the Colab VM, runs the supervised full-PANDORA export, and writes monitorable status files to Google Drive.

Default output: `/content/drive/MyDrive/Anay Agarwalla/Models/texttraits_full_export`

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/csboi/TextTraits.git"
BRANCH = "codex/pycharm-colab-setup"  # Change to main after this branch is merged.
REPO_DIR = Path("/content/TextTraits")
OUT_DIR = "/content/drive/MyDrive/Anay Agarwalla/Models/texttraits_full_export"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=False)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=False)
os.chdir(REPO_DIR)

print("Repo:", REPO_DIR)
print("Branch:", BRANCH)
print("Output:", OUT_DIR)
print("Commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)

## Smoke Test

Run this first to verify Drive paths, repo state, checkpoint writing, and monitor output without paying for a full training run.

In [ ]:
subprocess.run([
    "python",
    "training/colab_supervised_export.py",
    "run",
    "--out-dir", OUT_DIR,
    "--candidate-profile", "fast",
    "--selection-sample", "200000",
    "--max-rows", "500000",
    "--no-full-refit",
    "--heartbeat-seconds", "30",
    "--poll-seconds", "30",
], check=False)

## Full Export

Run this after the smoke test passes. It uses full PANDORA unless you add `--max-rows`.

In [ ]:
subprocess.run([
    "python",
    "training/colab_supervised_export.py",
    "run",
    "--out-dir", OUT_DIR,
    "--candidate-profile", "balanced",
    "--selection-sample", "500000",
    "--heartbeat-seconds", "60",
    "--poll-seconds", "60",
], check=False)

## Monitor Only

Run this if the notebook looks frozen or after reconnecting to the Colab runtime. It does not load PANDORA.

In [ ]:
subprocess.run([
    "python",
    "training/colab_supervised_export.py",
    "monitor",
    "--out-dir", OUT_DIR,
    "--watch",
    "--poll-seconds", "60",
], check=False)